In [1]:
# Install required libraries
#fatry:
import subprocess
import importlib

# Only install if not already installed
try:
    importlib.import_module('fastapi')
    importlib.import_module('uvicorn')
    print('fastapi and uvicorn already installed — skipping')
except ImportError:
    subprocess.run(['pip', 'install', 'fastapi', 'uvicorn[standard]'], check=True)

fastapi and uvicorn already installed — skipping


In [2]:
import os
import numpy as np
import json

# Point to the model folder 
MODEL_DIR = os.path.join(os.getcwd(), '..', 'model')
MODEL_DIR = os.path.abspath(MODEL_DIR)   # converts .. into a clean path
print("Model folder:", MODEL_DIR)
print()

# Check files exist 
required_files = ['model_weights.npz', 'scale_params.json']

for f in required_files:
    full_path = os.path.join(MODEL_DIR, f)
    if os.path.exists(full_path):
        print(f' {f} found')
    else:
        print(f' {f} NOT found at {full_path}')

# Load weights 
print()
weights = np.load(os.path.join(MODEL_DIR, 'model_weights.npz'))
print('Weight shapes:')
for name in weights.files:
    print(f'  {name}: {weights[name].shape}')

# Load scale params 
print()
with open(os.path.join(MODEL_DIR, 'scale_params.json'), 'r') as f:
    scale_params = json.load(f)
print('Features in scale_params.json:')
for feature, bounds in scale_params.items():
    print(f'  {feature}: min={bounds["min"]:.4f}, max={bounds["max"]:.4f}')

Model folder: /home/narjiss/Documents/PFE_Project/model

 model_weights.npz found
 scale_params.json found

Weight shapes:
  W1: (11, 16)
  b1: (1, 16)
  W2: (16, 1)
  b2: (1, 1)

Features in scale_params.json:
  CreditLineUsage: min=0.0000, max=10.0078
  Age: min=21.0000, max=109.0000
  Late30to59Days: min=0.0000, max=4.5951
  DebtRatio: min=0.0000, max=12.6346
  MonthlyIncome: min=0.0000, max=13.6352
  OpenCreditLines: min=0.0000, max=4.0604
  Late90Days: min=0.0000, max=4.5951
  RealEstateLoans: min=0.0000, max=3.4012
  Late60to89Days: min=0.0000, max=4.5951
  Dependents: min=0.0000, max=2.3979


In [3]:
# MinMax scaling function
# the same logic we ipplied during preprocessing in Document3, we gonna re-implement it here
# so the API can scale incoming raw inputs.

def scale_input(raw_input: dict, scale_params: dict) -> np.ndarray:
        # The exact order that the model expects (must match training column order)
        feature_order = [
            'CreditLineUsage', 'Age', 'Late30to59Days', 'DebtRatio',
            'MonthlyIncome', 'OpenCreditLines', 'Late90Days',
            'RealEstateLoans', 'Late60to89Days', 'Dependents',
            'MonthlyIncome_Was_Missing'
        ]

        scaled_values = []
      
        for feature in feature_order:
            raw_value = raw_input[feature]

            # MonthlyIncome_Was_Missing is already 0 or 1 — no scaling needed
            if feature == 'MonthlyIncome_Was_Missing':
                scaled_values.append(float(raw_value))
                continue

            f_min = scale_params[feature]['min']
            f_max = scale_params[feature]['max']

            # Apply MinMax formula
            scaled = (raw_value - f_min) / (f_max - f_min)

            # Clip to [0, 1] to handle values outside the training range
            scaled = float(np.clip(scaled, 0.0, 1.0))
            scaled_values.append(scaled)

        # Return as a 2D array with shape (1, 11)
        # Shape (1, 11) means: 1 sample, 11 features — matches training format
        return np.array(scaled_values).reshape(1, -1)


# ── Quick test ─────────────────────────────────────────────────────────────────
test_raw = {
    'CreditLineUsage': 0.5,
    'Age': 45,
    'Late30to59Days': 0,
    'DebtRatio': 0.3,
    'MonthlyIncome': 8.5,          # already log-transformed as in preprocessing
    'OpenCreditLines': 2.1,
    'Late90Days': 0,
    'RealEstateLoans': 0,
    'Late60to89Days': 0,
    'Dependents': 0,
    'MonthlyIncome_Was_Missing': 0
}

scaled = scale_input(test_raw, scale_params)
print('Scaled input shape:', scaled.shape)   # expect (1, 11)
print('Scaled values:', np.round(scaled, 4))

Scaled input shape: (1, 11)
Scaled values: [[0.05   0.2727 0.     0.0237 0.6234 0.5172 0.     0.     0.     0.
  0.    ]]


In [4]:
# Activation functions 
# Same Forward Propagation we  use on Document2 - same equation, same weights convention


def relu(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    Z = np.clip(Z, -500, 500)      # prevent overflow with extreme values
    return 1 / (1 + np.exp(-Z))


# Forward propagation (inference) 
def predict(X_scaled, W1, b1, W2, b2, threshold=0.4):

    # Hidden layer — MUST use np.dot(X, W1), not np.dot(W1, X)
    Z1 = np.dot(X_scaled, W1) + b1     # shape: (1, 16)
    A1 = relu(Z1)                       # shape: (1, 16)

    # Output layer
    Z2 = np.dot(A1, W2) + b2           # shape: (1, 1)
    A2 = sigmoid(Z2)                    # shape: (1, 1) — value in (0, 1)

    probability = float(A2[0, 0])
    verdict = 'HIGH RISK' if probability >= threshold else 'LOW RISK'

    return probability, verdict


# Quick test 
W1 = weights['W1']   # shape (11, 16)
b1 = weights['b1']   # shape (1, 16)
W2 = weights['W2']   # shape (16, 1)
b2 = weights['b2']   # shape (1, 1)

prob, verdict = predict(scaled, W1, b1, W2, b2)
print(f'Default probability : {prob:.4f}')
print(f'Risk verdict        : {verdict}')

Default probability : 0.2258
Risk verdict        : LOW RISK


In [5]:
# Write the API file to disk 
# We write the entire FastAPI app to a file called main.py.
# This is the file that uvicorn will run.

api_code = '''
"""Loan Default Risk Prediction API"""

import numpy as np
import json
from fastapi import FastAPI
from pydantic import BaseModel, Field

# ── Load model weights once at startup ────────────────────────────────────────
# Loading files is slow — we do it once when the server starts,
# not on every request.
weights      = np.load("model/model_weights.npz")
W1           = weights["W1"]   # (11, 16)
b1           = weights["b1"]   # (1, 16)
W2           = weights["W2"]   # (16, 1)
b2           = weights["b2"]   # (1, 1)

with open("model/scale_params.json", "r") as f:
    scale_params = json.load(f)

THRESHOLD = 0.4   # chosen in Document 2 for best F1 score

# ── Activation functions ───────────────────────────────────────────────────────
def relu(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    Z = np.clip(Z, -500, 500)
    return 1 / (1 + np.exp(-Z))

# ── Scaling ────────────────────────────────────────────────────────────────────
FEATURE_ORDER = [
    "CreditLineUsage", "Age", "Late30to59Days", "DebtRatio",
    "MonthlyIncome", "OpenCreditLines", "Late90Days",
    "RealEstateLoans", "Late60to89Days", "Dependents",
    "MonthlyIncome_Was_Missing"
]

def scale_input(raw: dict) -> np.ndarray:
    scaled = []
    for feature in FEATURE_ORDER:
        value = raw[feature]
        if feature == "MonthlyIncome_Was_Missing":
            scaled.append(float(value))
            continue
        f_min = scale_params[feature]["min"]
        f_max = scale_params[feature]["max"]
        v = float(np.clip((value - f_min) / (f_max - f_min), 0.0, 1.0))
        scaled.append(v)
    return np.array(scaled).reshape(1, -1)

# ── Inference ──────────────────────────────────────────────────────────────────
def forward(X):
    """Forward pass. X shape: (1, 11)."""
    Z1 = np.dot(X, W1) + b1   # (1, 16)
    A1 = relu(Z1)              # (1, 16)
    Z2 = np.dot(A1, W2) + b2  # (1, 1)
    A2 = sigmoid(Z2)           # (1, 1)
    return float(A2[0, 0])

# ── FastAPI app ────────────────────────────────────────────────────────────────
app = FastAPI(
    title="Loan Default Risk Prediction API",
    description="MLP built from scratch with NumPy — Narjiss Maimouni",
    version="1.0.0"
)

# ── Input schema ───────────────────────────────────────────────────────────────
# Pydantic validates every incoming request against this class.
# Field(...) marks a field as required and lets us add a description.
class ApplicantInput(BaseModel):
    CreditLineUsage        : float = Field(..., description="Ratio of credit used (raw, will be log-scaled internally)")
    Age                    : int   = Field(..., ge=18, le=120, description="Applicant age in years")
    Late30to59Days         : float = Field(..., ge=0,  description="Number of times 30-59 days late (raw count)")
    DebtRatio              : float = Field(..., ge=0,  description="Monthly debt payments / monthly gross income")
    MonthlyIncome          : float = Field(..., ge=0,  description="Monthly income — pass log1p value as used in preprocessing")
    OpenCreditLines        : float = Field(..., ge=0,  description="Number of open credit lines (raw count)")
    Late90Days             : float = Field(..., ge=0,  description="Number of times 90+ days late (raw count)")
    RealEstateLoans        : float = Field(..., ge=0,  description="Number of real estate loans (raw count)")
    Late60to89Days         : float = Field(..., ge=0,  description="Number of times 60-89 days late (raw count)")
    Dependents             : float = Field(..., ge=0,  description="Number of dependents (raw count)")
    MonthlyIncome_Was_Missing : int = Field(..., ge=0, le=1, description="1 if income was originally missing, 0 otherwise")

# ── Output schema ──────────────────────────────────────────────────────────────
class PredictionOutput(BaseModel):
    default_probability : float
    risk_verdict        : str
    threshold_used      : float

# ── Health check endpoint ──────────────────────────────────────────────────────
@app.get("/", summary="Health check")
def root():
    return {"message": "Loan Default Prediction API is running. Visit /docs to test it."}

# ── Prediction endpoint ────────────────────────────────────────────────────────
@app.post("/predict", response_model=PredictionOutput, summary="Predict default risk")
def predict(applicant: ApplicantInput):

    # Convert Pydantic model to plain dict
    raw = applicant.model_dump()

    # Scale the input
    X_scaled = scale_input(raw)

    # Run forward propagation
    probability = forward(X_scaled)

    # Apply threshold
    verdict = "HIGH RISK" if probability >= THRESHOLD else "LOW RISK"

    return PredictionOutput(
        default_probability=round(probability, 4),
        risk_verdict=verdict,
        threshold_used=THRESHOLD
    )
'''


# Define root path — one level up from notebooks/
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
MAIN_PY_PATH = os.path.join(ROOT_DIR, 'main.py')

with open(MAIN_PY_PATH, 'w') as f:
    f.write(api_code)

print('main.py written successfully')
print('Location:', MAIN_PY_PATH)
print('File size:', os.path.getsize(MAIN_PY_PATH), 'bytes')

main.py written successfully
Location: /home/narjiss/Documents/PFE_Project/main.py
File size: 5889 bytes


In [6]:
# Start the server
import subprocess
import time

# Root directory of the project
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Start the server as a background process
server_process = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '127.0.0.1', '--port', '8000', '--reload'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd=ROOT_DIR 
)

# Give the server a few seconds to start up
print('Starting server...')
time.sleep(4)

poll = server_process.poll()
if poll is None:
    print('Server started (PID:', server_process.pid, ')')
    print()
    print('API is now available at:')
    print('  http://127.0.0.1:8000')
    print('  http://127.0.0.1:8000/docs')
else:
    err = server_process.stderr.read().decode()
    print('Server failed to start')
    print(err)

Starting server...
Server started (PID: 6049 )

API is now available at:
  http://127.0.0.1:8000
  http://127.0.0.1:8000/docs


In [7]:
# Test the API with Python (requests library)

import requests
API_URL = 'http://127.0.0.1:8000/predict'

low_risk_applicant = {
    "CreditLineUsage": 0.3,
    "Age": 38,
    "Late30to59Days": 0,
    "DebtRatio": 0.2,
    "MonthlyIncome": 6000,
    "OpenCreditLines": 3,
    "Late90Days": 0,
    "RealEstateLoans": 1,
    "Late60to89Days": 0,
    "Dependents": 1,
    "MonthlyIncome_Was_Missing": 0
}

high_risk_applicant = {
    "CreditLineUsage": 0.95,
    "Age": 58,
    "Late30to59Days": 4,
    "DebtRatio": 0.9,
    "MonthlyIncome": 1200,
    "OpenCreditLines": 8,
    "Late90Days": 3,
    "RealEstateLoans": 0,
    "Late60to89Days": 2,
    "Dependents": 4,
    "MonthlyIncome_Was_Missing": 0
}

# Send requests and display results 
for label, applicant in [('Low Risk Applicant', low_risk_applicant),
                          ('High Risk Applicant', high_risk_applicant)]:
    response = requests.post(API_URL, json=applicant)

    if response.status_code == 200:
        result = response.json()
        print(f'── {label} ──────────────────────────')
        print(f'  Default Probability : {result["default_probability"]:.4f}')
        print(f'  Risk Verdict        : {result["risk_verdict"]}')
        print(f'  Threshold Used      : {result["threshold_used"]}')
        print()
    else:
        print(f'❌ Error {response.status_code}: {response.text}')

── Low Risk Applicant ──────────────────────────
  Default Probability : 0.1823
  Risk Verdict        : LOW RISK
  Threshold Used      : 0.4

── High Risk Applicant ──────────────────────────
  Default Probability : 0.9299
  Risk Verdict        : HIGH RISK
  Threshold Used      : 0.4

